# Notebook 05: Character Transformer Evaluation & Game Playback
We evaluate the trained Character Transformer on unseen test secrets using constrained decoding:
$$P(\text{word} | \text{game\_state})$$

In [ ]:
import sys
import os
import zipfile

# Auto-extract zip file if Kaggle uploaded dataset as a raw zip
if os.path.exists("/kaggle/input"):
    for root_dir, dirs, files in os.walk("/kaggle/input"):
        for file in files:
            if file.endswith(".zip") and "wordle" in file.lower():
                zip_path = os.path.join(root_dir, file)
                print(f"Extracting uploaded Kaggle dataset: {zip_path} -> /kaggle/working/")
                with zipfile.ZipFile(zip_path, 'r') as zip_ref:
                    zip_ref.extractall("/kaggle/working/")

# Dynamic path resolution for local and Kaggle environments
possible_roots = [
    ".",
    "..",
    "/kaggle/working",
    "/kaggle/working/wordle",
    "/kaggle/working/wordle-ai",
    "/kaggle/input/wordle",
    "/kaggle/input/wordle/wordle",
    "/kaggle/input/wordle/wordle-ai",
    "/kaggle/input/wordle-ai",
]

# Search recursively inside /kaggle/input for any folder containing 'src'
if os.path.exists("/kaggle/input"):
    for root_dir, dirs, files in os.walk("/kaggle/input"):
        if "src" in dirs:
            possible_roots.insert(0, root_dir)

for root in possible_roots:
    if os.path.exists(os.path.join(root, "src")):
        abs_root = os.path.abspath(root)
        if abs_root not in sys.path:
            sys.path.insert(0, abs_root)
        print(f"Found 'src' package at: {abs_root}")
        break

if os.path.exists("/kaggle/input"):
    print("Running on Kaggle GPU/CPU Environment!")
else:
    print("Running on Local Environment!")

from src.wordle.dictionary import WordleDictionary
from src.solvers.character_transformer_solver import CharacterTransformerSolver
from src.evaluation.benchmark import BenchmarkRunner

dict_mgr = WordleDictionary()
solver = CharacterTransformerSolver(dict_mgr)
runner = BenchmarkRunner(dict_mgr)

metrics = runner.evaluate_solver(solver, test_secrets=dict_mgr.test_secrets[:25])
print(f"Char Transformer Win Rate: {metrics.win_rate:.1f}%, Avg Guesses: {metrics.avg_guesses:.2f}")